In [4]:
import torch
import torch.nn as nn
import math # Added for positional encoding

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Definition of PositionalEncoding and MiniTransformer from previous context
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        encoder_layers = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

        # Ensure causal masking
        self.register_buffer('src_mask', None)

    def _generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, src):
        if self.src_mask is None or self.src_mask.size(0) != src.size(1):
            self.src_mask = self._generate_square_subsequent_mask(src.size(1)).to(src.device)

        src = self.embedding(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        output = self.transformer_encoder(src, mask=self.src_mask) # Pass the mask to the encoder
        output = self.fc_out(output)
        return output

# Code to create data.txt (as requested by the user within this cell)
sample_text = """\nOlá! Esta é uma demonstração de um MiniTransformer.\nEle aprende a gerar texto baseado nos exemplos que você fornece.\nQuanto mais texto, melhor será o aprendizado e a qualidade da geração.\nVamos usar um texto um pouco mais longo para ver um resultado melhor.\nEspero que goste dos resultados após o treinamento.\nA inteligência artificial está avançando rapidamente e é fascinante ver como modelos como este podem criar conteúdo novo.\nPara que um modelo de linguagem como este MiniTransformer possa gerar texto mais coerente e relevante, ele precisa ser exposto a uma quantidade substancial de dados textuais. Esses dados servem como a base para o modelo aprender padrões, gramática, vocabulário e até mesmo nuances de estilo. Um conjunto de dados pequeno pode levar a um aprendizado limitado, resultando em saídas repetitivas ou sem sentido. Por outro lado, um conjunto de dados diversificado e extenso permite que o modelo capture uma gama maior de contextos e estruturas linguísticas. Isso é fundamental para que a geração de texto seja mais fluida e se assemelhe à linguagem humana. Aumentar o tamanho do texto de entrada é uma etapa crucial para melhorar a performance de qualquer modelo de processamento de linguagem natural.\n"""

with open("data.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

# ----------------------------
# Ler dataset
# ----------------------------

with open("data.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))

stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for ch, i in stoi.items()}

vocab_size = len(chars)

# ----------------------------
# Encode
# ----------------------------

data = torch.tensor(
    [stoi[c] for c in text],
    dtype=torch.long
).to(device) # Move data to device

# ----------------------------
# Mini batches
# ----------------------------

block_size = 64 # Increased from 32
batch_size = 16 # Increased from 8

def get_batch():

    ix = torch.randint(
        len(data) - block_size - 1,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i+block_size]
        for i in ix
    ])

    y = torch.stack([
        data[i+1:i+block_size+1]
        for i in ix
    ])

    return x.to(device), y.to(device) # Move batches to device

# ----------------------------
# Modelo
# ----------------------------

# Increased model capacity and training epochs for better generation
model = MiniTransformer(
    vocab_size,
    d_model=256,  # Increased from 128
    nhead=8,
    num_layers=4,
    dim_feedforward=4096 # Increased from 2048
).to(device) # Move model to device

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

criterion = nn.CrossEntropyLoss()

# ----------------------------
# Treino
# ----------------------------

epochs = 5000 # Changed from 10000 as requested

for epoch in range(epochs):

    x, y = get_batch()

    logits = model(x)

    loss = criterion(
        logits.view(-1, vocab_size),
        y.view(-1)
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch} | Loss {loss.item():.4f}"
        )

# ----------------------------
# Geração
# ----------------------------

# Start generation with a more meaningful phrase
start_text = "A inteligência"
context = torch.tensor([list(map(lambda char: stoi[char], start_text))]).to(device) # Move context to device
generated = start_text

for _ in range(50):

    logits = model(context)

    probs = torch.softmax(
        logits[:, -1, :],
        dim=-1
    )

    next_token = torch.multinomial(
        probs,
        num_samples=1
    )

    context = torch.cat(
        [context, next_token],
        dim=1
    )

    generated += itos[next_token.item()] # Ensure next_token is on CPU for string conversion

print("\nTexto gerado:")
print(generated)

Using device: cuda
Epoch 0 | Loss 4.1267
Epoch 100 | Loss 2.9381
Epoch 200 | Loss 2.9315
Epoch 300 | Loss 2.5175
Epoch 400 | Loss 2.1797
Epoch 500 | Loss 2.0976
Epoch 600 | Loss 1.7226
Epoch 700 | Loss 1.3193
Epoch 800 | Loss 0.7817
Epoch 900 | Loss 0.5936
Epoch 1000 | Loss 0.4414
Epoch 1100 | Loss 0.3370
Epoch 1200 | Loss 0.3041
Epoch 1300 | Loss 0.2864
Epoch 1400 | Loss 0.2301
Epoch 1500 | Loss 0.2461
Epoch 1600 | Loss 0.1944
Epoch 1700 | Loss 0.2185
Epoch 1800 | Loss 0.2311
Epoch 1900 | Loss 0.2052
Epoch 2000 | Loss 0.1802
Epoch 2100 | Loss 0.1479
Epoch 2200 | Loss 0.1714
Epoch 2300 | Loss 0.1530
Epoch 2400 | Loss 0.1662
Epoch 2500 | Loss 0.1535
Epoch 2600 | Loss 0.1661
Epoch 2700 | Loss 0.1694
Epoch 2800 | Loss 0.1338
Epoch 2900 | Loss 0.1136
Epoch 3000 | Loss 0.1760
Epoch 3100 | Loss 0.1837
Epoch 3200 | Loss 0.1308
Epoch 3300 | Loss 0.1360
Epoch 3400 | Loss 0.1105
Epoch 3500 | Loss 0.1285
Epoch 3600 | Loss 0.1181
Epoch 3700 | Loss 0.1301
Epoch 3800 | Loss 0.1143
Epoch 3900 | Loss 